# L1 — Rapport de profiling des 2 datasets

## Projet E-commerce — Google Colab + Google Drive

Ce notebook est adapté à ton dossier Google Drive :

```text
/content/drive/MyDrive/projet_ecommerce_l1
```

Il réalise le livrable **L1 : Rapport de profiling des 2 datasets**.

À la fin, le notebook génère dans le dossier `reports` :

- `profiling_online_retail.html`
- `profiling_shipping.html`
- `profiling_summary_l1.csv`
- `missing_values_online_retail.csv`
- `missing_values_shipping.csv`
- `outliers_online_retail.csv`
- `outliers_shipping.csv`
- `quality_checks_online_retail.csv`
- `quality_checks_shipping.csv`
- `L1_synthese_profiling.md`
- `L1_livrables_profiling.zip`

## Structure utilisée

```text
projet_ecommerce_l1/
├── data/
│   └── raw/
│       ├── online_retail.csv
│       └── ecommerce_shipping.csv
└── reports/
```

# 0. Installation des librairies

On installe les librairies nécessaires pour lire les fichiers, analyser les données et générer les rapports HTML.

> Si `ydata-profiling` affiche un avertissement de dépréciation, ce n'est pas grave. Le rapport peut quand même être généré.

In [ ]:
!pip install -q pandas numpy openpyxl ydata-profiling ipywidgets tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.6 MB/s eta 0:00:00


# 1. Importation des librairies

Cette section importe les librairies nécessaires.

In [ ]:
import os
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from google.colab import drive, files
from IPython.display import display

# Import de ydata-profiling.
# Le warning de dépréciation n'est pas bloquant.
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    from ydata_profiling import ProfileReport

# Options d'affichage pandas
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

print("Librairies importées avec succès.")

Librairies importées avec succès.


# 2. Connexion à Google Drive

On connecte Colab à Google Drive pour que les fichiers générés soient sauvegardés directement dans ton Drive.

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


# 3. Configuration des dossiers du projet

Le notebook travaille directement dans ton dossier Drive :

```text
Mon Drive / projet_ecommerce_l1
```

In [ ]:
# Dossier principal dans Google Drive
PROJECT_DIR = Path("/content/drive/MyDrive/projet_ecommerce_l1")

# Dossier des données brutes
DATA_DIR = PROJECT_DIR / "data" / "raw"

# Dossier des rapports
REPORT_DIR = PROJECT_DIR / "reports"

# Création des dossiers si nécessaire
DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Dossier projet        :", PROJECT_DIR)
print("Dossier données brutes:", DATA_DIR)
print("Dossier rapports      :", REPORT_DIR)

Dossier projet        : /content/drive/MyDrive/projet_ecommerce_l1
Dossier données brutes: /content/drive/MyDrive/projet_ecommerce_l1/data/raw
Dossier rapports      : /content/drive/MyDrive/projet_ecommerce_l1/reports


# 4. Vérifier les fichiers présents dans `data/raw`

Cette étape permet de vérifier si les fichiers CSV ou Excel sont déjà présents dans Drive.

In [ ]:
def list_files(folder: Path):
    # Affiche tous les fichiers présents dans un dossier.
    all_files = sorted([p for p in folder.glob("**/*") if p.is_file()])

    if not all_files:
        print(f"Aucun fichier trouvé dans : {folder}")
        return []

    print(f"Fichiers trouvés dans {folder} :")
    for file in all_files:
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"- {file.relative_to(folder)} ({size_mb:.2f} MB)")
    return all_files

raw_files = list_files(DATA_DIR)

Fichiers trouvés dans /content/drive/MyDrive/projet_ecommerce_l1/data/raw :
- Train.csv (0.42 MB)
- data.csv (43.47 MB)


# 5. Importer les datasets si nécessaire

Si les datasets ne sont pas encore dans `data/raw`, mets `UPLOAD_FILES = True` puis exécute la cellule.

Les fichiers seront copiés automatiquement dans :

```text
/content/drive/MyDrive/projet_ecommerce_l1/data/raw
```

In [ ]:
UPLOAD_FILES = False  # Mets True seulement si tu veux importer les fichiers depuis ton PC.

if UPLOAD_FILES:
    uploaded = files.upload()

    for filename in uploaded.keys():
        source_path = Path(filename)
        destination_path = DATA_DIR / filename
        shutil.move(str(source_path), str(destination_path))
        print(f"Fichier importé dans Drive : {destination_path}")
else:
    print("Upload désactivé. Si besoin, mets UPLOAD_FILES = True.")

Upload désactivé. Si besoin, mets UPLOAD_FILES = True.


# 6. Fonction robuste pour lire les fichiers

Le dataset Online Retail peut contenir des caractères spéciaux dans `Description`.

Pour éviter les erreurs d'encodage, la fonction ci-dessous teste automatiquement plusieurs encodages :

- `utf-8`
- `utf-8-sig`
- `ISO-8859-1`
- `latin1`
- `cp1252`

Elle peut aussi lire les fichiers Excel `.xlsx` ou `.xls`.

In [ ]:
def read_table_safely(file_path: Path) -> pd.DataFrame:
    # Lit un fichier CSV ou Excel en testant plusieurs encodages si nécessaire.
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()

    if suffix in [".xlsx", ".xls"]:
        print(f"Lecture Excel : {file_path.name}")
        return pd.read_excel(file_path)

    if suffix != ".csv":
        raise ValueError(f"Format non supporté : {file_path.suffix}. Utilise CSV ou Excel.")

    encodings = ["utf-8", "utf-8-sig", "ISO-8859-1", "latin1", "cp1252"]
    separators = [",", ";", "	"]
    last_error = None

    for encoding in encodings:
        for sep in separators:
            try:
                df = pd.read_csv(file_path, encoding=encoding, sep=sep)

                # Si une seule colonne est détectée, le séparateur est probablement incorrect.
                if df.shape[1] <= 1 and sep != separators[-1]:
                    continue

                print(f"Lecture réussie : {file_path.name}")
                print(f"Encodage utilisé : {encoding}")
                print(f"Séparateur utilisé : {repr(sep)}")
                return df

            except Exception as e:
                last_error = e

    raise RuntimeError(f"Impossible de lire {file_path.name}. Dernière erreur : {last_error}")

# 7. Détection automatique des deux datasets

Le notebook essaie de détecter automatiquement :

- **Online Retail** avec les colonnes `InvoiceNo`, `StockCode`, `Quantity`, `UnitPrice` ;
- **Shipping** avec les colonnes `Mode_of_Shipment`, `Discount_offered`, `Weight_in_gms`.

Si la détection échoue, tu peux renseigner les fichiers manuellement dans la cellule suivante.

In [ ]:
# Liste des fichiers CSV ou Excel disponibles dans data/raw
candidate_files = sorted(
    [p for p in DATA_DIR.glob("*") if p.suffix.lower() in [".csv", ".xlsx", ".xls"]]
)

print("Fichiers candidats :")
for p in candidate_files:
    print("-", p.name)

ONLINE_REQUIRED_COLUMNS = {"InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate", "UnitPrice"}
SHIPPING_REQUIRED_COLUMNS = {"Mode_of_Shipment", "Discount_offered", "Cost_of_the_Product", "Weight_in_gms", "Reached.on.Time_Y.N"}

online_file = None
shipping_file = None

# Détection par colonnes
for file_path in candidate_files:
    try:
        sample_df = read_table_safely(file_path).head(10)
        cols = set(sample_df.columns)

        if ONLINE_REQUIRED_COLUMNS.issubset(cols):
            online_file = file_path

        if SHIPPING_REQUIRED_COLUMNS.issubset(cols):
            shipping_file = file_path

    except Exception as e:
        print(f"Impossible de prévisualiser {file_path.name}: {e}")

print("Détection automatique :")
print("Online Retail détecté :", online_file.name if online_file else "Non détecté")
print("Shipping détecté      :", shipping_file.name if shipping_file else "Non détecté")

Fichiers candidats :
- Train.csv
- data.csv
Lecture réussie : Train.csv
Encodage utilisé : utf-8
Séparateur utilisé : ','
Lecture réussie : data.csv
Encodage utilisé : ISO-8859-1
Séparateur utilisé : ','
Détection automatique :
Online Retail détecté : data.csv
Shipping détecté      : Train.csv


## 7.1 Option manuelle si la détection échoue

Si un fichier n'est pas détecté, décommente les lignes suivantes et adapte les noms.

Exemples possibles :

```python
online_file = DATA_DIR / "online_retail.csv"
shipping_file = DATA_DIR / "ecommerce_shipping.csv"
```

In [ ]:
# online_file = DATA_DIR / "online_retail.csv"
# shipping_file = DATA_DIR / "ecommerce_shipping.csv"

if online_file is None:
    raise FileNotFoundError("Dataset Online Retail non détecté. Renseigne online_file manuellement.")

if shipping_file is None:
    raise FileNotFoundError("Dataset Shipping non détecté. Renseigne shipping_file manuellement.")

print("Fichier Online Retail utilisé :", online_file)
print("Fichier Shipping utilisé      :", shipping_file)

Fichier Online Retail utilisé : /content/drive/MyDrive/projet_ecommerce_l1/data/raw/data.csv
Fichier Shipping utilisé      : /content/drive/MyDrive/projet_ecommerce_l1/data/raw/Train.csv


# 8. Chargement des deux datasets

On charge les deux datasets dans des DataFrames pandas :

- `online_retail`
- `shipping`

À ce stade, les données restent brutes. On ne nettoie rien dans L1.

In [ ]:
online_retail = read_table_safely(online_file)
shipping = read_table_safely(shipping_file)

print("Online Retail :", online_retail.shape)
print("Shipping      :", shipping.shape)

Lecture réussie : data.csv
Encodage utilisé : ISO-8859-1
Séparateur utilisé : ','
Lecture réussie : Train.csv
Encodage utilisé : utf-8
Séparateur utilisé : ','
Online Retail : (541909, 8)
Shipping      : (10999, 12)


# 9. Aperçu des données

On affiche les premières lignes pour comprendre la structure des datasets.

In [ ]:
print("Aperçu Online Retail")
display(online_retail.head())

print("Aperçu Shipping")
display(shipping.head())

Aperçu Online Retail


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


Aperçu Shipping


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1


# 10. Dimensions des datasets

On calcule le nombre de lignes et de colonnes pour chaque dataset.

In [ ]:
shape_summary = pd.DataFrame({
    "dataset": ["Online Retail", "E-Commerce Shipping"],
    "nombre_lignes": [online_retail.shape[0], shipping.shape[0]],
    "nombre_colonnes": [online_retail.shape[1], shipping.shape[1]]
})

shape_summary

,dataset,nombre_lignes,nombre_colonnes
0,Online Retail,541909,8
1,E-Commerce Shipping,10999,12


# 11. Analyse des types de colonnes

Cette partie analyse toutes les colonnes : type, nombre de valeurs non nulles, valeurs nulles, valeurs uniques et exemple.

In [ ]:
def dataframe_schema(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    # Crée un dictionnaire de données technique simple.
    rows = []

    for col in df.columns:
        example_value = df[col].dropna().iloc[0] if df[col].notna().any() else np.nan

        rows.append({
            "dataset": dataset_name,
            "column": col,
            "dtype": str(df[col].dtype),
            "non_null_count": df[col].notna().sum(),
            "null_count": df[col].isna().sum(),
            "unique_count": df[col].nunique(dropna=True),
            "example_value": example_value
        })

    return pd.DataFrame(rows)

schema_online = dataframe_schema(online_retail, "Online Retail")
schema_shipping = dataframe_schema(shipping, "E-Commerce Shipping")

print("Schéma Online Retail")
display(schema_online)

print("Schéma Shipping")
display(schema_shipping)

Schéma Online Retail


,dataset,column,dtype,non_null_count,null_count,unique_count,example_value
0,Online Retail,InvoiceNo,object,541909,0,25900,536365
1,Online Retail,StockCode,object,541909,0,4070,85123A
2,Online Retail,Description,object,540455,1454,4223,WHITE HANGING HEART T-LIGHT HOLDER
3,Online Retail,Quantity,int64,541909,0,722,6
4,Online Retail,InvoiceDate,object,541909,0,23260,12/1/2010 8:26
5,Online Retail,UnitPrice,float64,541909,0,1630,2.55
6,Online Retail,CustomerID,float64,406829,135080,4372,17850.0
7,Online Retail,Country,object,541909,0,38,United Kingdom


Schéma Shipping


,dataset,column,dtype,non_null_count,null_count,unique_count,example_value
0,E-Commerce Shipping,ID,int64,10999,0,10999,1
1,E-Commerce Shipping,Warehouse_block,object,10999,0,5,D
2,E-Commerce Shipping,Mode_of_Shipment,object,10999,0,3,Flight
3,E-Commerce Shipping,Customer_care_calls,int64,10999,0,6,4
4,E-Commerce Shipping,Customer_rating,int64,10999,0,5,2
5,E-Commerce Shipping,Cost_of_the_Product,int64,10999,0,215,177
6,E-Commerce Shipping,Prior_purchases,int64,10999,0,8,3
7,E-Commerce Shipping,Product_importance,object,10999,0,3,low
8,E-Commerce Shipping,Gender,object,10999,0,2,F
9,E-Commerce Shipping,Discount_offered,int64,10999,0,65,44


# 12. Taux de nullité

Le taux de nullité est un critère obligatoire du livrable L1.

Il indique le pourcentage de valeurs manquantes par colonne.

In [ ]:
def missing_values_report(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    # Calcule le nombre et le taux de valeurs manquantes.
    report = pd.DataFrame({
        "dataset": dataset_name,
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_rate_%": (df.isna().sum().values / len(df) * 100).round(2),
        "non_missing_count": df.notna().sum().values
    })

    return report.sort_values("missing_rate_%", ascending=False)

missing_online = missing_values_report(online_retail, "Online Retail")
missing_shipping = missing_values_report(shipping, "E-Commerce Shipping")

print("Taux de nullité — Online Retail")
display(missing_online)

print("Taux de nullité — Shipping")
display(missing_shipping)

In [ ]:
# Sauvegarde dans Google Drive
missing_online.to_csv(REPORT_DIR / "missing_values_online_retail.csv", index=False)
missing_shipping.to_csv(REPORT_DIR / "missing_values_shipping.csv", index=False)

print("Fichiers sauvegardés :")
print(REPORT_DIR / "missing_values_online_retail.csv")
print(REPORT_DIR / "missing_values_shipping.csv")

# 13. Identification des doublons

On identifie les lignes totalement dupliquées.

Dans L1, on identifie seulement les doublons. Leur suppression sera décidée dans le nettoyage Silver.

In [ ]:
def duplicate_report(df: pd.DataFrame, dataset_name: str) -> dict:
    # Calcule le nombre et le taux de doublons.
    duplicate_count = df.duplicated().sum()
    duplicate_rate = round(duplicate_count / len(df) * 100, 2)

    return {
        "dataset": dataset_name,
        "duplicate_count": int(duplicate_count),
        "duplicate_rate_%": duplicate_rate
    }

duplicates_online = duplicate_report(online_retail, "Online Retail")
duplicates_shipping = duplicate_report(shipping, "E-Commerce Shipping")

duplicates_summary = pd.DataFrame([duplicates_online, duplicates_shipping])
duplicates_summary

In [ ]:
# Exemples de doublons, si disponibles
print("Exemples de doublons Online Retail")
display(online_retail[online_retail.duplicated(keep=False)].head(10))

print("Exemples de doublons Shipping")
display(shipping[shipping.duplicated(keep=False)].head(10))

# 14. Statistiques descriptives

On affiche les statistiques principales des colonnes numériques : minimum, maximum, moyenne, quartiles, etc.

In [ ]:
print("Statistiques numériques — Online Retail")
display(online_retail.describe(include=[np.number]).T)

print("Statistiques numériques — Shipping")
display(shipping.describe(include=[np.number]).T)

# 15. Analyse des variables catégorielles

On analyse les colonnes non numériques : nombre de valeurs uniques, valeur la plus fréquente et taux de dominance.

In [ ]:
def categorical_report(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    # Analyse simple des colonnes catégorielles/textuelles.
    cat_cols = df.select_dtypes(exclude=[np.number]).columns
    rows = []

    for col in cat_cols:
        mode_values = df[col].mode(dropna=True)
        top_value = mode_values.iloc[0] if not mode_values.empty else np.nan
        top_count = (df[col] == top_value).sum() if pd.notna(top_value) else 0

        rows.append({
            "dataset": dataset_name,
            "column": col,
            "unique_count": df[col].nunique(dropna=True),
            "top_value": top_value,
            "top_count": int(top_count),
            "top_rate_%": round(top_count / len(df) * 100, 2)
        })

    if not rows:
        return pd.DataFrame(columns=["dataset", "column", "unique_count", "top_value", "top_count", "top_rate_%"])

    return pd.DataFrame(rows).sort_values("unique_count", ascending=False)

categorical_online = categorical_report(online_retail, "Online Retail")
categorical_shipping = categorical_report(shipping, "E-Commerce Shipping")

print("Variables catégorielles — Online Retail")
display(categorical_online)

print("Variables catégorielles — Shipping")
display(categorical_shipping)

# 16. Détection des outliers avec la méthode IQR

La méthode IQR permet d'identifier les valeurs extrêmes.

Formule :

```text
IQR = Q3 - Q1
borne basse = Q1 - 1.5 × IQR
borne haute = Q3 + 1.5 × IQR
```

In [ ]:
def detect_outliers_iqr(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    # Détecte les outliers pour toutes les colonnes numériques.
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    results = []

    for col in numeric_cols:
        series = df[col].dropna()

        if series.empty:
            continue

        Q1 = series.quantile(0.25)
        Q3 = series.quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
        outlier_count = outlier_mask.sum()
        outlier_rate = round(outlier_count / len(df) * 100, 2)

        results.append({
            "dataset": dataset_name,
            "column": col,
            "Q1": Q1,
            "Q3": Q3,
            "IQR": IQR,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "outlier_count": int(outlier_count),
            "outlier_rate_%": outlier_rate,
            "min_value": series.min(),
            "max_value": series.max()
        })

    if not results:
        return pd.DataFrame(columns=["dataset", "column", "Q1", "Q3", "IQR", "lower_bound", "upper_bound", "outlier_count", "outlier_rate_%"])

    return pd.DataFrame(results).sort_values("outlier_rate_%", ascending=False)

outliers_online = detect_outliers_iqr(online_retail, "Online Retail")
outliers_shipping = detect_outliers_iqr(shipping, "E-Commerce Shipping")

print("Outliers — Online Retail")
display(outliers_online)

print("Outliers — Shipping")
display(outliers_shipping)

In [ ]:
# Sauvegarde des rapports d'outliers
outliers_online.to_csv(REPORT_DIR / "outliers_online_retail.csv", index=False)
outliers_shipping.to_csv(REPORT_DIR / "outliers_shipping.csv", index=False)

print("Fichiers sauvegardés :")
print(REPORT_DIR / "outliers_online_retail.csv")
print(REPORT_DIR / "outliers_shipping.csv")

# 17. Contrôles spécifiques — Online Retail

On vérifie les problèmes importants du dataset transactionnel :

- factures commençant par `C` ;
- quantités négatives ;
- prix unitaires nuls ou négatifs ;
- descriptions manquantes ;
- `CustomerID` manquant ;
- codes produits spéciaux.

In [ ]:
def online_retail_quality_checks(df: pd.DataFrame) -> pd.DataFrame:
    checks = []

    def add_check(name, count, comment):
        checks.append({
            "controle": name,
            "count": int(count),
            "rate_%": round(count / len(df) * 100, 2),
            "commentaire": comment
        })

    if "InvoiceNo" in df.columns:
        count_returns = df["InvoiceNo"].astype(str).str.startswith("C").sum()
        add_check("Factures commençant par C", count_returns, "Retours ou annulations probables.")

    if "Quantity" in df.columns:
        add_check("Quantity < 0", (df["Quantity"] < 0).sum(), "Quantités négatives liées souvent aux retours.")
        add_check("Quantity = 0", (df["Quantity"] == 0).sum(), "Quantités nulles à examiner.")

    if "UnitPrice" in df.columns:
        add_check("UnitPrice <= 0", (df["UnitPrice"] <= 0).sum(), "Prix nul ou négatif à traiter.")

    if "Description" in df.columns:
        add_check("Description manquante", df["Description"].isna().sum(), "Description produit absente.")

    if "CustomerID" in df.columns:
        add_check("CustomerID manquant", df["CustomerID"].isna().sum(), "Important pour RFM/CLV.")

    if "StockCode" in df.columns:
        special_codes = ["POST", "D", "C2", "M", "BANK CHARGES"]
        count_special_codes = df["StockCode"].astype(str).isin(special_codes).sum()
        add_check("Codes produits spéciaux", count_special_codes, "Codes non produits à traiter dans Silver.")

    return pd.DataFrame(checks)

online_checks = online_retail_quality_checks(online_retail)
display(online_checks)

online_checks.to_csv(REPORT_DIR / "quality_checks_online_retail.csv", index=False)
print("Fichier sauvegardé :", REPORT_DIR / "quality_checks_online_retail.csv")

# 18. Contrôles spécifiques — E-Commerce Shipping

On vérifie les colonnes utiles pour ST4 :

- remise offerte ;
- coût produit ;
- poids ;
- note client ;
- livraison à temps.

In [ ]:
def shipping_quality_checks(df: pd.DataFrame) -> pd.DataFrame:
    checks = []

    def add_check(name, count, comment):
        checks.append({
            "controle": name,
            "count": int(count),
            "rate_%": round(count / len(df) * 100, 2),
            "commentaire": comment
        })

    if "Discount_offered" in df.columns:
        add_check("Discount_offered < 0", (df["Discount_offered"] < 0).sum(), "Remise négative incohérente.")
        add_check("Discount_offered > 100", (df["Discount_offered"] > 100).sum(), "Remise supérieure à 100 % incohérente.")

    if "Cost_of_the_Product" in df.columns:
        add_check("Cost_of_the_Product <= 0", (df["Cost_of_the_Product"] <= 0).sum(), "Coût produit nul ou négatif.")

    if "Weight_in_gms" in df.columns:
        add_check("Weight_in_gms <= 0", (df["Weight_in_gms"] <= 0).sum(), "Poids nul ou négatif incohérent.")

    if "Customer_rating" in df.columns:
        add_check("Customer_rating hors 1-5", (~df["Customer_rating"].between(1, 5)).sum(), "Note client hors plage attendue.")

    if "Reached.on.Time_Y.N" in df.columns:
        add_check("Reached.on.Time_Y.N hors 0/1", (~df["Reached.on.Time_Y.N"].isin([0, 1])).sum(), "Variable binaire attendue.")

    return pd.DataFrame(checks)

shipping_checks = shipping_quality_checks(shipping)
display(shipping_checks)

shipping_checks.to_csv(REPORT_DIR / "quality_checks_shipping.csv", index=False)
print("Fichier sauvegardé :", REPORT_DIR / "quality_checks_shipping.csv")

# 19. Distribution des principales variables

Cette partie donne une vision rapide des valeurs les plus fréquentes dans quelques colonnes métier.

In [ ]:
if "Country" in online_retail.columns:
    print("Top 10 pays — Online Retail")
    display(online_retail["Country"].value_counts(dropna=False).head(10).to_frame("count"))

if "Mode_of_Shipment" in shipping.columns:
    print("Modes de livraison — Shipping")
    display(shipping["Mode_of_Shipment"].value_counts(dropna=False).to_frame("count"))

if "Product_importance" in shipping.columns:
    print("Importance produit — Shipping")
    display(shipping["Product_importance"].value_counts(dropna=False).to_frame("count"))

# 20. Résumé global L1

On crée un tableau de synthèse qui résume le profiling des deux datasets.

In [ ]:
summary_l1 = pd.DataFrame({
    "dataset": ["Online Retail", "E-Commerce Shipping"],
    "nombre_lignes": [online_retail.shape[0], shipping.shape[0]],
    "nombre_colonnes": [online_retail.shape[1], shipping.shape[1]],
    "colonnes_avec_nulls": [
        int((online_retail.isna().sum() > 0).sum()),
        int((shipping.isna().sum() > 0).sum())
    ],
    "total_valeurs_nulles": [
        int(online_retail.isna().sum().sum()),
        int(shipping.isna().sum().sum())
    ],
    "nombre_doublons": [
        int(online_retail.duplicated().sum()),
        int(shipping.duplicated().sum())
    ],
    "taux_doublons_%": [
        round(online_retail.duplicated().sum() / len(online_retail) * 100, 2),
        round(shipping.duplicated().sum() / len(shipping) * 100, 2)
    ],
    "colonnes_numeriques": [
        int(len(online_retail.select_dtypes(include=[np.number]).columns)),
        int(len(shipping.select_dtypes(include=[np.number]).columns))
    ],
    "colonnes_categorielles": [
        int(len(online_retail.select_dtypes(exclude=[np.number]).columns)),
        int(len(shipping.select_dtypes(exclude=[np.number]).columns))
    ]
})

summary_l1

In [ ]:
# Sauvegarde des fichiers de synthèse
summary_l1.to_csv(REPORT_DIR / "profiling_summary_l1.csv", index=False)
schema_online.to_csv(REPORT_DIR / "schema_online_retail.csv", index=False)
schema_shipping.to_csv(REPORT_DIR / "schema_shipping.csv", index=False)

if not categorical_online.empty:
    categorical_online.to_csv(REPORT_DIR / "categorical_online_retail.csv", index=False)

if not categorical_shipping.empty:
    categorical_shipping.to_csv(REPORT_DIR / "categorical_shipping.csv", index=False)

print("Fichiers de synthèse sauvegardés dans :", REPORT_DIR)

# 21. Génération des rapports HTML avec `ydata-profiling`

Cette étape génère les deux rapports HTML demandés pour L1.

Les rapports seront sauvegardés ici :

```text
/content/drive/MyDrive/projet_ecommerce_l1/reports
```

> Cette partie peut prendre du temps, surtout pour Online Retail.

In [ ]:
# minimal=True accélère la génération et réduit les risques de crash mémoire dans Colab.
PROFILE_MINIMAL = True

profile_online = ProfileReport(
    online_retail,
    title="L1 Profiling Report - Online Retail",
    explorative=True,
    minimal=PROFILE_MINIMAL
)

online_profile_path = REPORT_DIR / "profiling_online_retail.html"
profile_online.to_file(online_profile_path)

print("Rapport Online Retail généré :", online_profile_path)

In [ ]:
profile_shipping = ProfileReport(
    shipping,
    title="L1 Profiling Report - E-Commerce Shipping",
    explorative=True,
    minimal=PROFILE_MINIMAL
)

shipping_profile_path = REPORT_DIR / "profiling_shipping.html"
profile_shipping.to_file(shipping_profile_path)

print("Rapport Shipping généré :", shipping_profile_path)

# 22. Création d'une synthèse Markdown

Cette cellule crée un fichier `L1_synthese_profiling.md` exploitable dans Git ou dans le rapport final.

In [ ]:
def markdown_table_from_df(df: pd.DataFrame) -> str:
    return df.to_markdown(index=False)

synthese_md = f"""
# L1 — Synthèse du profiling des datasets

## Dossier projet

`{PROJECT_DIR}`

## Datasets analysés

- Online Retail : `{online_file.name}`
- E-Commerce Shipping : `{shipping_file.name}`

## Résumé global

{markdown_table_from_df(summary_l1)}

## Fichiers générés

- `profiling_online_retail.html`
- `profiling_shipping.html`
- `profiling_summary_l1.csv`
- `missing_values_online_retail.csv`
- `missing_values_shipping.csv`
- `outliers_online_retail.csv`
- `outliers_shipping.csv`
- `quality_checks_online_retail.csv`
- `quality_checks_shipping.csv`
- `schema_online_retail.csv`
- `schema_shipping.csv`

## Contrôles réalisés

- Dimensions des datasets
- Types des colonnes
- Taux de nullité
- Doublons
- Statistiques descriptives
- Outliers avec la méthode IQR
- Contrôles spécifiques Online Retail
- Contrôles spécifiques Shipping

## Remarque

Ce livrable L1 sert à observer la qualité des données brutes.
Les décisions de nettoyage seront documentées dans le livrable L2 `DECISIONS.md` et appliquées dans le pipeline Bronze vers Silver.
"""

synthese_path = REPORT_DIR / "L1_synthese_profiling.md"
synthese_path.write_text(synthese_md, encoding="utf-8")

print("Synthèse Markdown générée :", synthese_path)

# 23. Vérification finale des fichiers générés

On vérifie que tous les fichiers L1 sont bien présents dans le dossier `reports`.

In [ ]:
reports_files = list_files(REPORT_DIR)

# 24. Création d'une archive ZIP des livrables L1

Cette archive contient tous les fichiers générés dans `reports`.

In [ ]:
zip_base = PROJECT_DIR / "L1_livrables_profiling"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=REPORT_DIR)

print("Archive ZIP créée :", zip_path)

# 25. Télécharger l'archive ZIP sur ton ordinateur

Cette étape est optionnelle, car le ZIP est déjà sauvegardé dans Google Drive.

In [ ]:
DOWNLOAD_ZIP = False  # Mets True si tu veux télécharger le ZIP sur ton ordinateur.

if DOWNLOAD_ZIP:
    files.download(zip_path)
else:
    print("Téléchargement désactivé. Mets DOWNLOAD_ZIP = True si nécessaire.")

26. Nettoyer

In [ ]:
import pandas as pd

# Charger les données brutes
df = pd.read_excel("Online Retail.xlsx", sheet_name="Online Retail")
print(f"Brut : {df.shape[0]:,} lignes")

# 1. Séparer les retours des ventes
df_returns = df[df['InvoiceNo'].astype(str).str.startswith('C')].copy()
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')].copy()
print(f"Retours extraits : {df_returns.shape[0]:,} lignes")

# 2. Supprimer les lignes sans CustomerID
df = df.dropna(subset=['CustomerID'])
print(f"Après suppression nulls CustomerID : {df.shape[0]:,} lignes")

# 3. Supprimer les codes SKU spéciaux
codes_speciaux = ['POST', 'D', 'C2', 'M', 'BANK CHARGES', 'AMAZONFEE', 'DOT']
df = df[~df['StockCode'].astype(str).str.upper().isin(codes_speciaux)]
print(f"Après suppression SKU spéciaux : {df.shape[0]:,} lignes")

# 4. Supprimer prix nuls ou négatifs
df = df[df['UnitPrice'] > 0]
print(f"Après suppression prix <= 0 : {df.shape[0]:,} lignes")

# 5. Supprimer quantités nulles ou négatives
df = df[df['Quantity'] > 0]
print(f"Après suppression quantités <= 0 : {df.shape[0]:,} lignes")

# 6. Convertir InvoiceDate en datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# 7. Convertir CustomerID en entier
df['CustomerID'] = df['CustomerID'].astype(int)

# 8. Ajouter colonne TotalRevenue
df['TotalRevenue'] = df['Quantity'] * df['UnitPrice']

print(f"\n✅ Dataset UCI nettoyé : {df.shape[0]:,} lignes | {df.shape[1]} colonnes")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Online Retail.xlsx'

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# 1. Définition des chemins du projet dans Google Drive
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/projet_ecommerce_l1")

RAW_DIR = PROJECT_DIR / "data" / "raw"
SILVER_DIR = PROJECT_DIR / "data" / "silver"
REPORT_DIR = PROJECT_DIR / "reports"

SILVER_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Dossier projet :", PROJECT_DIR)
print("Dossier raw :", RAW_DIR)
print("Dossier silver :", SILVER_DIR)
print("Dossier rapports :", REPORT_DIR)

Dossier projet : /content/drive/MyDrive/projet_ecommerce_l1
Dossier raw : /content/drive/MyDrive/projet_ecommerce_l1/data/raw
Dossier silver : /content/drive/MyDrive/projet_ecommerce_l1/data/silver
Dossier rapports : /content/drive/MyDrive/projet_ecommerce_l1/reports


In [11]:


online_returns_clean = online_raw[mask_return].copy()

online_returns_clean["invoice_no"] = online_returns_clean["InvoiceNo_str"]
online_returns_clean["stock_code"] = online_returns_clean["StockCode_str"]
online_returns_clean["description_clean"] = online_returns_clean["Description"].astype(str).str.strip().str.upper()
online_returns_clean["quantity_original"] = online_returns_clean["Quantity_num"]
online_returns_clean["returned_quantity"] = online_returns_clean["Quantity_num"].abs()
online_returns_clean["unit_price"] = online_returns_clean["UnitPrice_num"]
online_returns_clean["invoice_date"] = online_returns_clean["InvoiceDate_parsed"]
online_returns_clean["return_value"] = online_returns_clean["returned_quantity"] * online_returns_clean["unit_price"]



mask_rejected_for_sales = (
    mask_return |
    mask_invalid_quantity |
    mask_invalid_price |
    mask_missing_description |
    mask_special_code |
    mask_invalid_date |
    mask_duplicate
)

online_rejected = online_raw[mask_rejected_for_sales].copy()

def rejection_reason_online(row):
    reasons = []

    if str(row["InvoiceNo_str"]).upper().startswith("C") or row["Quantity_num"] < 0:
        reasons.append("retour_ou_annulation")
    if pd.isna(row["Quantity_num"]) or row["Quantity_num"] <= 0:
        reasons.append("quantite_invalide")
    if pd.isna(row["UnitPrice_num"]) or row["UnitPrice_num"] <= 0:
        reasons.append("prix_invalide")
    if pd.isna(row["Description"]) or str(row["Description"]).strip() == "" or str(row["Description"]).strip().lower() == "nan":
        reasons.append("description_manquante")
    if row["StockCode_str"] in special_codes:
        reasons.append("code_produit_special")
    if pd.isna(row["InvoiceDate_parsed"]):
        reasons.append("date_invalide")

    return " | ".join(reasons) if reasons else "doublon_exact"

online_rejected["rejection_reason"] = online_rejected.apply(rejection_reason_online, axis=1)

# ------------------------------------------------------------
# 5.3 Table des ventes propres
# ------------------------------------------------------------
# Règles :
# - pas de retour ;
# - quantité positive ;
# - prix positif ;
# - description présente ;
# - date valide ;
# - pas de code spécial ;
# - pas de doublon exact.

online_sales_clean = online_raw[~mask_rejected_for_sales].copy()

online_sales_clean = online_sales_clean.rename(columns={
    "InvoiceNo": "invoice_no_original",
    "StockCode": "stock_code_original",
    "Description": "description_original",
    "Quantity": "quantity_original",
    "InvoiceDate": "invoice_date_original",
    "UnitPrice": "unit_price_original",
    "CustomerID": "customer_id_original",
    "Country": "country_original"
})

online_sales_clean["invoice_no"] = online_sales_clean["InvoiceNo_str"]
online_sales_clean["stock_code"] = online_sales_clean["StockCode_str"]
online_sales_clean["description_clean"] = online_sales_clean["description_original"].astype(str).str.strip().str.upper()
online_sales_clean["quantity"] = online_sales_clean["Quantity_num"]
online_sales_clean["unit_price"] = online_sales_clean["UnitPrice_num"]
online_sales_clean["invoice_date"] = online_sales_clean["InvoiceDate_parsed"]
online_sales_clean["revenue"] = online_sales_clean["quantity"] * online_sales_clean["unit_price"]

# CustomerID n'est pas obligatoire pour ST3.
# On le garde quand il existe.
if "customer_id_original" in online_sales_clean.columns:
    online_sales_clean["customer_id"] = pd.to_numeric(online_sales_clean["customer_id_original"], errors="coerce")
else:
    online_sales_clean["customer_id"] = np.nan

online_sales_clean["country"] = online_sales_clean["country_original"].astype(str).str.strip()

# Variables temporelles utiles pour ST3
online_sales_clean["year"] = online_sales_clean["invoice_date"].dt.year
online_sales_clean["month"] = online_sales_clean["invoice_date"].dt.month
online_sales_clean["year_month"] = online_sales_clean["invoice_date"].dt.to_period("M").astype(str)
online_sales_clean["week"] = online_sales_clean["invoice_date"].dt.isocalendar().week.astype(int)
online_sales_clean["day_name"] = online_sales_clean["invoice_date"].dt.day_name()
online_sales_clean["hour"] = online_sales_clean["invoice_date"].dt.hour

# On ajoute des flags d'outliers, mais on ne supprime pas automatiquement les outliers.
online_sales_clean = add_iqr_outlier_flags(
    online_sales_clean,
    columns=["quantity", "unit_price", "revenue"]
)

print("Lignes Online Retail initiales :", len(online_raw))
print("Ventes propres :", len(online_sales_clean))
print("Retours isolés :", len(online_returns_clean))
print("Lignes rejetées pour ventes :", len(online_rejected))

Dossier projet : /content/drive/MyDrive/projet_ecommerce_l1
Dossier raw : /content/drive/MyDrive/projet_ecommerce_l1/data/raw
Dossier silver : /content/drive/MyDrive/projet_ecommerce_l1/data/silver
Dossier rapports : /content/drive/MyDrive/projet_ecommerce_l1/reports
Fichiers CSV trouvés dans data/raw :


FileNotFoundError: Dataset Online Retail non trouvé. Vérifie que le fichier est bien dans data/raw.

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

# Chemins de ton projet dans Google Drive
PROJECT_DIR = Path("/content/drive/MyDrive/projet_ecommerce_l1")
RAW_DIR = PROJECT_DIR / "data" / "raw"

print("Dossier raw :", RAW_DIR)

# Fonction de lecture robuste pour éviter les problèmes d'encodage
def read_csv_safely(path, nrows=None):
    encodings = ["utf-8", "utf-8-sig", "ISO-8859-1", "latin1", "cp1252"]

    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc, nrows=nrows)
            print(f"Lecture réussie : {path.name} avec encodage {enc}")
            return df
        except Exception:
            continue

    raise ValueError(f"Impossible de lire le fichier : {path.name}")


# Vérifier les fichiers CSV disponibles
csv_files = list(RAW_DIR.glob("*.csv"))

print("Fichiers CSV trouvés :")
for file in csv_files:
    print("-", file.name)


# Si shipping n'existe pas, on essaie de le détecter automatiquement
if "shipping" not in globals():

    shipping_file = None

    for file in csv_files:
        try:
            sample = read_csv_safely(file, nrows=5)
            cols = set(sample.columns)

            # Colonnes attendues dans le dataset Shipping
            required_shipping_cols = {
                "Mode_of_Shipment",
                "Discount_offered",
                "Cost_of_the_Product",
                "Weight_in_gms"
            }

            if required_shipping_cols.issubset(cols):
                shipping_file = file
                break

        except Exception as e:
            print(f"Erreur lecture {file.name} :", e)

    if shipping_file is None:
        raise FileNotFoundError(
            "Dataset Shipping non trouvé. Vérifie que le fichier Shipping est bien dans data/raw."
        )

    # Chargement complet du dataset Shipping
    shipping = read_csv_safely(shipping_file)
    print("Dataset Shipping chargé avec succès :", shipping_file.name)

else:
    print("La variable shipping existe déjà.")

print("Dimensions du dataset Shipping :", shipping.shape)
shipping.head()

Dossier raw : /content/drive/MyDrive/projet_ecommerce_l1/data/raw
Fichiers CSV trouvés :


FileNotFoundError: Dataset Shipping non trouvé. Vérifie que le fichier Shipping est bien dans data/raw.